In [ ]:
from pathlib import Path
import pandas as pd
import re
import datetime

# This is the journal file path you want to parse the info
file_path = 'journal-parsing'

blk_list = []
blk_size = 0

with open(file_path, 'rb') as f:
    # read the 's_blocksize' part to determine the journal block size
    f.seek(12)
    size_data = f.read(4)
    blk_size = int(size_data.hex(), 16)
    
    f.seek(0)
    while True:
        blk = f.read(blk_size)
        if not blk:
            break
        blk = blk.hex()
        blk_list.append(blk)

blk_size, blk_list

In [ ]:
HEAD = "c03b3998" # magic number


first_block = 0
first_seq = 0
jb_list = [] # journal block list

# bitmask flag
compat_flags = {
    0x1: "JBD2_FEATURE_COMPAT_CHECKSUM"
}
incompat_flags = {
    0x1: "JBD2_FEATURE_INCOMPAT_REVOKE",
    0x2: "JBD2_FEATURE_INCOMPAT_64BIT",
    0x4: "JBD2_FEATURE_INCOMPAT_ASYNC_COMMIT",
    0x8: "JBD2_FEATURE_INCOMPAT_CSUM_V2",
    0x10: "JBD2_FEATURE_INCOMPAT_CSUM_V3",
    0x20: "JBD2_FEATURE_INCOMPAT_FAST_COMMIT"
}
tag_flags = {
    0x1: "JBD2_FLAG_ESCAPE", # on-disk block is escaped
    0x2: "JBD2_FLAG_SAME_UUID", # block has same uuid as previous
    0x4: "JBD2_FLAG_DELETED", # block deleted by this transaction
    0x8: "JBD2_FLAG_LAST_TAG" # last tag in this descriptor block
}

compat_list = []
incompat_list = []

for i in range(len(blk_list)):
    blk = blk_list[i]
    # transaction = {} # transaction 단위로 묶기 위해 사용
    # data_blocks_list = [] # data 블록들만 한번에 묶기 위해 사용
    
    if i == 0: # superblock
        # check flags
        compat_f = int(blk[72:80], 16)
        incompat_f = int(blk[80:88], 16)
        
        compat_list = [
            v for k, v in compat_flags.items() if compat_f & k
        ]
        incompat_list = [
            v for k, v in incompat_flags.items() if incompat_f & k
        ]

        first_block = int(blk[40:48], 16)
        first_seq = int(blk[48:56], 16)

        # superblock info
        jb_list.append({
            "s_blocksize": blk_size,
            "s_maxlen": int(blk[32:40], 16),
            "s_first": first_block,
            "s_sequence": first_seq,
            "s_start": int(blk[56:64], 16)
        })
    
    elif blk[:8] == HEAD: # descriptor, commit, revoke block
        blocktype = int(blk[8:16], 16)
        seq = int(blk[16:32], 16)
        if blocktype == 1: # descriptor
            block_list = []
            if "JBD2_FEATURE_INCOMPAT_CSUM_V3" in incompat_list:
                if "JBD2_FEATURE_INCOMPAT_64BIT" in incompat_list:
                    uuid = ""
                    for j in range((8192-24)//64):
                        tag = blk[24+j*32:24+(j+1)*32]
                        t_blocknr = tag[:8] # lower 32-bits
                        t_blocknr_high = tag[16:24] # upper 32-bits
                        blocknr = int(t_blocknr_high+t_blocknr, 16)
                        t_flags = int(tag[8:16], 16)
                        t_checksum = tag[24:32]
                        
                        tag_list = [
                            v for k, v in tag_flags.items() if t_flags & k
                        ]
                        
                        if not "JBD2_FLAG_SAME_UUID" in tag_list:
                            uuid = blk[24+(j+1)*32:24+(j+2)*32]
                            j += 1
                            
                        
                        block_list.append({
                            "t_blocknr": t_blocknr,
                            "t_blocknr_high": t_blocknr_high,
                            "blocknr": blocknr,
                            "tag_list": tag_list,
                            "t_checksum": t_checksum,
                            "uuid": uuid,
                        })
                        
                        if "JBD2_FLAG_LAST_TAG" in tag_list:
                            break
                else:
                    for j in range((8192-24)//32):
                        tag = blk[24+j*32:24+(j+1)*24]
                        t_blocknr = tag[:8]
                        blocknr = int(t_blocknr, 16)
                        t_flags = int(tag[8:16], 16)
                        t_checksum = tag[24:32]
                        
                        tag_list = [
                            v for k, v in tag_flags.items() if t_flags & k
                        ]
                        
                        block_list.append({
                            "t_blocknr": t_blocknr,
                            "t_blocknr_high": t_blocknr_high,
                            "blocknr": blocknr,
                            "tag_list": tag_list,
                            "t_checksum": t_checksum
                        })
                        
                        if "JBD2_FLAG_LAST_TAG" in tag_list:
                            break
            # else # if "JBD2_FEATURE_INCOMPAT_CSUM_V3" is NOT set
                
            jb_list.append({
                "type": "descriptor",
                "sequence": seq, 
                "block_count": len(block_list),
                "block_list": block_list,
            })
        elif blocktype == 2: # commit
            
            # time stamp 파싱
            timestamp = int(blk[96:112], 16)
            time = datetime.datetime.fromtimestamp(timestamp)
            nsec = int(str(int(blk[112:120], 16))[:3])
            time = str(time)+'.'+str(nsec)
            
            jb_list.append({
                "type": "commit",
                "time": time,
            })
        elif blk[8:16] == 5: # revoke
            rb = {
                
            }
    else: # data block
        jb_list.append(blk)
        

# #for j in range(len(blk_list)):
# blk = blk_list

# if j == 0: # 슈퍼블록
#     print(blk)
#     first_seq = int(blk[48:56], 16)
    
#     print("first seq:", first_seq)
    
# else:
#     if blk[:8] == HEAD: # descriptor, commmit, revoke 블록
#         blk_type = int(blk[8:16], 16)
#         if blk_type == 1: # descriptor 블록
#             seq = int(blk[16:24], 16)
#             block_nrs = []
#             for k in range((8192-24)//32): # 블록의 끝까지 반복하면서 블록 넘버 뽑아내기
#                 tag = blk[24+k*32:24+(k+1)*32]
#                 blk_nr = int(tag[:8], 16)
#                 block_nrs.append(blk_nr)
#             transaction = {"seq": seq, "block_nrs": block_nrs}
#             data_blocks = []
#             # print(transaction)
#         elif blk_type == 2: # commit 블록
#             if int(blk[16:24], 16) == seq: # descriptor - commit pair가 맞으면
#                 timestamp = int(blk[96:112], 16)
#                 time = datetime.datetime.fromtimestamp(timestamp)
#                 nsec = int(str(int(blk[112:120], 16))[:3])
#                 time = str(time)+'.'+str(nsec)
#                 transaction.update({"time": time, "blocks": data_blocks})
#                 transaction["block_nrs"] = transaction["block_nrs"][:len(data_blocks)]
#                 jb_list.append(transaction)
#                 # print(transaction)
#         # elif blk_type == 5: # revoke 블록
#     else: # metadata block
#         # print(blk)
#         data_blocks.append(blk)
                
    
# df = pd.DataFrame(jb_list)
# # df_time_sorted = df.sort_values(by=["time"], ascending=True)
# df.to_csv(bin_path_list[0].name[:-3]+"csv", index=False)

jb_list